# 25 · Multi-agent failure modes: loops, lost context and guards
Level L5 · Part 5 · core · ~25 min · builds on 24

## Goal
Reproduce a loop between two agents with a real model, then stop it with three guards: a call budget, a handoff
limit and a repeat check on the route. You end with a run that stops, and a reason you can log.

## Why it matters
Two agents that each think the other owns a missing fact pass the case back and forth. Each turn is a model call
and a bill, and nothing in the code ends the run. A person finds out later, from the invoice or the customer.

| No guards | With three guards |
|---|---|
| The case goes back and forth until someone stops it | A budget, a handoff limit and a repeat check stop it |
| The only record is a chat history with no reason for the stop | Every stop has a reason that the run can log |
| Cost grows with each turn, with no ceiling | At most the budget, then a person takes over |

**What it makes easy:** stopping a bad run in code and handing the case to a person with the reason attached.
**When you don't need it:** a fixed workflow with known steps (lesson 11), or one agent with a step cap (lesson 06).

## The idea
A **loop** between agents is a **handoff** cycle: agent A hands the case to agent B, B hands it back, and nothing new
is learned on the way. The usual cause is a **gap**: a fact that no agent can get, but each one is told to ask the
other. Here the refund desk's rule needs a carrier scan date. The order record does not hold one (only status,
days_late and total), so the order desk can never supply it and the refund desk can never decide.

Multi-agent systems fail in other ways too: a note that drops the context the next agent needed (**lost context**),
and agents that never learn that the job is over (the paper's FM-1.5, in At work). A **guard** is a check in your code, outside the model, that stops
or refuses a step when a limit is hit. Three guards stop a run. A **call budget** caps the model calls per case. A
**handoff limit** caps the handoffs. A **repeat check** reads the **route**, the list of `(from, to)` handoff pairs so
far (the shared state both desks write to), and refuses a handoff the route already has.

Analogy: two shop counters that send a customer back and forth. A rule that the customer goes to a manager after a set
number of visits ends the walk without anyone needing to notice.

## Picture
The order desk and the refund desk each need a fact the other one holds. Without a guard, the route never ends.

```mermaid
flowchart LR
  C[Customer: order 1002, wants money back] --> O[Order desk]
  O -->|hand_off to refund| R[Refund desk]
  R -->|hand_off: need carrier scan date| O
  O -->|hand_off: scan not in the order record| R
  R -.->|no end: each desk waits for the other| O
```

## Step by step
**Step 1 · setup and the data.** The key is read from `.env`, never typed here. While a recording replays, a placeholder
is used and no key is sent. `support_eval.json` is a hand-written file of fictional orders and the returns policy, the
same file lessons 10 to 14 and 21 use, and lesson 24. It holds no personal data. Order 1002 is in transit and four days late.
The policy says refunds are for lost orders or damaged deliveries, and anything else goes to a person.

In [1]:
import sys; sys.path.insert(0, "..")
import json
from pathlib import Path
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("25_multi-agent-failure-modes")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"   # the course's pinned model, as in lessons 02 to 24
PRICE_IN, PRICE_OUT = 0.25, 1.50  # USD per 1M tokens, as checked for lesson 02 on 2026-10-20
client = genai.Client(  # in your own project keep vertexai=True and pass your own key
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx
)
GOLDEN = json.loads(Path("../data/support_eval.json").read_text())  # fictional orders and the returns policy
ORDERS, POLICY, ORDER_ID = GOLDEN["world"]["orders"], GOLDEN["policy"], "1002"  # order 1002: the one the customer names
MSG = "My order 1002 is four days late. I want my money back."
print("order 1002 keys:", list(ORDERS[ORDER_ID]))  # no carrier scan date in the record

LLM mode: replay (25_multi-agent-failure-modes.yaml)


order 1002 keys: ['status', 'days_late', 'total']


**Step 2 · one helper for the calls.** `ask` makes one awaited call and logs its tokens and cost, so Step 7 can add up
each run. Nothing here calls the model. The token counts and dollar costs are tiny here, but multiply them by thousands
of cases and many turns. The notebook allows top-level `await`, so the awaited calls run directly in a cell. `llm_replay`
comes from lesson 02, and the async client (`client.aio`) from lesson 12.

In [2]:
LOG = []  # every model call this lesson makes: (step, input tokens, output tokens, USD)


async def ask(step: str, contents, config: types.GenerateContentConfig):
    """One awaited model call, logged so the cost of each run can be added up."""
    resp = await client.aio.models.generate_content(model=MODEL, contents=contents, config=config)
    u = resp.usage_metadata
    out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    LOG.append((step, u.prompt_token_count, out, (u.prompt_token_count * PRICE_IN + out * PRICE_OUT) / 1_000_000))
    return resp

**Step 3 · two desks and one tool each.** Each desk has its own brief, and its only tool is `hand_off`: it passes the
case and a note to the other desk. The harness puts the facts a desk needs in its prompt (the order record for the
order desk, the policy for the refund desk), so no desk spends a call on a lookup. The order record has no carrier scan
date, and the refund desk's rule needs one. That gap is written into the briefs on purpose, because real loops come
from gaps like it.

In [3]:
def lookup_order(order_id: str) -> dict:
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


HAND_OFF = types.FunctionDeclaration(
    name="hand_off", description="Pass the case to the other desk with a note of what it needs.",
    parameters_json_schema={"type": "object", "properties": {
        "to": {"type": "string", "enum": ["order", "refund"]},
        "note": {"type": "string", "description": "One or two sentences: the order, what you know, what is open."}},
        "required": ["to", "note"]})
ORDER_BRIEF = ("You are the order desk. You read order records; you cannot see carrier scans and you cannot decide refunds. "
               "If the customer asks for money back, or the refund desk asks for something, reply with a hand_off to "
               "'refund' that carries the order facts. If the refund desk asks for a carrier scan date, say the scan is not "
               "in the order record, and hand off to 'refund'.")
REFUND_BRIEF = ("You are the refund desk. You decide refunds under the policy. Our desk rule: no refund or refusal is "
                "decided until the carrier scan date is known, and you cannot see it. If the note does not give it, "
                "hand off to 'order' and ask for it. Only when it is given, decide and answer the customer in one sentence.")
DESK_BRIEF = {"order": ORDER_BRIEF, "refund": REFUND_BRIEF}; print("desks:", list(DESK_BRIEF), "| tool:", HAND_OFF.name)

desks: ['order', 'refund'] | tool: hand_off


**Step 4 · one turn and the route.** A turn is one model call by one desk, and the reply must be a `hand_off` or a
one-sentence answer. The cache is there only to avoid paying twice: an identical prompt returns the stored answer, so no
call is repeated. The route (defined in The idea) is the list of handoffs so far. The guards are code, not prompt text.
`check_handoff` decides whether a handoff may pass, and `run_desks` checks the call budget before each call. With
`guards=None` no guard runs; the `stop_after` cap is the lesson's own stop, so the notebook ends.

In [4]:
def tools_config(*decls, brief: str) -> types.GenerateContentConfig:
    """One desk's settings: its brief and only the tools it may use."""
    return types.GenerateContentConfig(
        system_instruction=brief, temperature=0, tools=[types.Tool(function_declarations=list(decls))],
        tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="AUTO")))


def desk_prompt(desk: str, note: str) -> str:
    """What a desk reads: the message, its desk's facts and the other desk's note ('(none...)' = the first desk, no note)."""
    facts = f"Order record: {lookup_order(ORDER_ID)}" if desk == "order" else f"Policy: {POLICY}"
    return f"Customer message: {MSG}\n{facts}\nNote from the other desk: {note or '(none: you are the first desk)'}"


CACHE, CACHE_HITS = {}, []  # key (desk, whole prompt): a new note is a new prompt, so a new call; value (hand_off, answer)


async def take_turn(step: str, desk: str, note: str):
    """One desk's turn: one model call, or the stored answer for an identical prompt. Returns (hand_off or None, text)."""
    prompt = desk_prompt(desk, note)
    if (desk, prompt) in CACHE:
        CACHE_HITS.append(step)  # no model call: same prompt, same temperature-0 answer as before
        return CACHE[(desk, prompt)]
    resp = await ask(step, prompt, tools_config(HAND_OFF, brief=DESK_BRIEF[desk]))
    parts = resp.candidates[0].content.parts
    hand = next((dict(p.function_call.args) for p in parts
                 if p.function_call and p.function_call.name == "hand_off"), None)
    answer = "".join(p.text or "" for p in parts)
    if hand is None and not answer:
        raise RuntimeError(f"{step}: the desk gave neither a hand_off nor an answer")
    CACHE[(desk, prompt)] = (hand, answer)
    return hand, answer

**The guards.** `check_handoff` runs before each handoff is passed on. It reads the route and refuses the handoff when
a guard fires: the handoff limit first, then the repeat check. The call budget is checked in `run_desks`, before each call.

In [5]:
def check_handoff(attempts: list, frm: str, to: str, guards: dict) -> str | None:
    """The guards for one handoff. Returns the reason to refuse it, or None to allow it."""
    done = [(a["from"], a["to"]) for a in attempts if not a["refused"]]  # passed-on handoffs; refused ones are not in it
    if len(done) + 1 > guards["max_handoffs"]:  # + 1 counts this handoff too, before it is passed on
        return f"handoff limit {guards['max_handoffs']} reached"
    if guards["repeat_check"] and (frm, to) in done:   # the repeat check: the route already has this handoff
        return "the same handoff again"
    return None


async def run_desks(label: str, guards: dict | None, stop_after: int):
    """Pass the case between the desks until one answers, a guard refuses, or the lesson's cap ends the run.
    'Escalated to a person' is only text here: no person is notified in this lab, the run just stops."""
    attempts, desk, note = [], "order", ""
    for n in range(1, stop_after + 1):
        if guards and n > guards["max_calls"]:   # the call budget, checked before each call
            return attempts, f"call budget of {guards['max_calls']} used up: escalated to a person"
        hand, answer = await take_turn(f"{label}{n} {desk}", desk, note)
        if hand is None:
            return attempts, f"answered the customer: {answer}"
        verdict = check_handoff(attempts, desk, hand["to"], guards) if guards else None
        attempts.append({"from": desk, "to": hand["to"], "note": hand["note"], "refused": verdict})
        if verdict:
            return attempts, f"guard refused the handoff ({verdict}): escalated to a person"
        desk, note = hand["to"], hand["note"]
    return attempts, f"the lesson stopped the run after {stop_after} calls; no guard was on"

**Step 5 · reproduce the loop (3 calls, no guard).** The run goes order, refund, order, and then the lesson's cap
stops it. Each note is the only context the next desk gets. Read the notes: the refund desk asks for the scan, and
the order desk says the scan is not in the record. A real system would keep going. Each call costs a fraction of a cent
here, but multiply it by thousands of cases and many turns.

In [6]:
A_ATTEMPTS, A_END = await run_desks("A", guards=None, stop_after=3)
for a in A_ATTEMPTS:
    print(f"{a['from']:>6} -> {a['to']:<6} note: {a['note']}")
print("run A:", A_END)

 order -> refund note: The customer is requesting a refund for order 1002, which is currently in transit and 4 days late. The order total is $45.00.
refund -> order  note: The customer is requesting a refund for order 1002, which is 4 days late; please provide the carrier scan date so I can determine if a refund is applicable.
 order -> refund note: The carrier scan date is not in the order record for order 1002, which is 4 days late. The customer is requesting a refund.
run A: the lesson stopped the run after 3 calls; no guard was on


**Step 6 · the same case with three guards.** The guards are a call budget of 2, a handoff limit of 1 and the repeat
check (the route may not repeat a handoff). Run B starts from the same message and gets the same prompts as A1 and A2.
Their answers are cached, so B replays them and makes zero new calls. That is a lesson shortcut, not something guards
do. The first handoff is allowed. The second is refused by the handoff limit, the only guard that fires here. The repeat
check does not fire, because refund to order is not yet on the route. The call budget does not fire either: the run
ends after 2 calls, so run A's third call never happens. The budget would fire on a third call; a tiny run could show it.

In [7]:
GUARDS = {"max_calls": 2, "max_handoffs": 1, "repeat_check": True}
B_ATTEMPTS, B_END = await run_desks("B-replay-", guards=GUARDS, stop_after=5)
for a in B_ATTEMPTS:
    print(f"{a['from']:>6} -> {a['to']:<6} " + (f"refused: {a['refused']}" if a["refused"] else "allowed"))
print("run B:", B_END, "| turns replayed from run A:", CACHE_HITS)

 order -> refund allowed
refund -> order  refused: handoff limit 1 reached
run B: guard refused the handoff (handoff limit 1 reached): escalated to a person | turns replayed from run A: ['B-replay-1 order', 'B-replay-2 refund']


**Step 7 · what each run cost (no call).** Every call is in `LOG`, so the cost of each run is a sum. Run A's cost is
the bill for a loop that had no stop; run B added no call, because its turns came from the stored answers.

In [8]:
for label, name in (("A", "run A, no guard"), ("B", "run B, guards")):
    rows = [r for r in LOG if r[0].startswith(label)]
    print(f"{name:16} calls {len(rows)}  input tokens {sum(r[1] for r in rows):>5}  ${sum(r[3] for r in rows):.6f}")
llm_replay.stop()  # the last model call of this notebook

run A, no guard  calls 3  input tokens   681  $0.000422
run B, guards    calls 0  input tokens     0  $0.000000


**Reading the output.** Run A made 3 calls and stopped only because the lesson stopped it. Run B made no new call: its
two turns were replayed from run A's stored answers. Only its handoff limit fired, on the second handoff; the repeat
check and the call budget stayed quiet. The Try it shows the repeat check firing on run A's route. The "escalated to a
person" text is only a string: no person is notified in this lab, the run just stops. The route in run A is the evidence of the loop: its third
handoff (order to refund) repeats its first. Notice what the notes carry. A desk never sees the earlier notes, so
anything left out of the note is lost to the next desk.

## At work
- The MAST study of multi-agent failures (arXiv 2503.13657) reports "1600+ annotated traces collected across 7 popular
  MAS frameworks" and "14 unique modes, clustered into 3 categories". Two of them, as the paper names them: FM-1.3
  "Step repetition" and FM-1.5 "Unaware of termination conditions" (the agents that never learn the job is over).
  → lesson 29 (traces show which agent repeated which step).
- Anthropic's multi-agent research post (2025-06-13) reports multi-agent runs at about 15 times the tokens of a chat turn,
  so a run without a limit is an unbounded bill. → lesson 33 (caps and routing).
- Production runs cap steps and spend per run, and send the case to a person with the reason logged.
  → lesson 32 (retries, timeouts and idempotency) and lesson 31 (the audit log).

Go deeper: framework step limits (LangGraph's recursion limit) do the call budget's job; check the name in its docs.

## Common mistakes
1. **A guard read under a misspelled key.** `guards.get("max_calls")` would return `None` for a misspelled key, so the
   guard is silently off. Plain `[]` fails loudly. The cell below raises on purpose, with no model call.

In [9]:
GUARDS_TYPO = {"max_call": 2, "max_handoffs": 1, "repeat_check": True}  # the key is "max_calls", not "max_call"
print(GUARDS_TYPO["max_calls"])  # KeyError: the budget name is wrong, so the run would have had no budget

KeyError: 'max_calls'

The fix: name each guard once, in one dict, and read it with `[]` so a typo stops the run before any call is made.

2. **Checking the budget after the call.** A check placed after `ask` lets one more call through. Put it before the call,
   as `run_desks` does, and count the calls that were made.
3. **Trusting the brief to stop the loop.** "Hand off only once" in a prompt is a wish, not a guard. The limit is code.
4. **Passing a note that drops facts.** The next desk sees only the note. Put the order number, what is known and what is
   still open into every hand-off.

## Production checklist
- Set a call budget, a spend budget and a handoff limit per case, and log which one stopped the run (lesson 33).
- Keep the route as the shared state, and refuse a handoff that repeats one already made.
- Escalate to a person with the case, the route and the reason, not with a bare error.
- Trace every desk's turn with its name, so a loop can be found in one trace (lesson 29).
- Test the guards with a case that loops on purpose, in CI (lesson 28).

## Try it
Write `loop_start(route)` that takes the route, a list of `(from, to)` pairs, and returns the index of the first handoff
that repeats an earlier pair, or `None` if no pair repeats. No model call is needed: the routes below are made up.
Then call it on the route of run A from Step 5. The solution also runs the repeat check from Step 4 on that route.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
def loop_start(route: list[tuple[str, str]]) -> int | None:
    seen = set()
    for i, pair in enumerate(route):
        if pair in seen:
            return i
        seen.add(pair)
    return None


print(loop_start([("order", "refund"), ("refund", "order")]))                 # None: no repeat
print(loop_start([("order", "refund"), ("refund", "order"), ("order", "refund")]))  # 2: the third handoff repeats the first
print("run A route:", loop_start([(a["from"], a["to"]) for a in A_ATTEMPTS]))  # 2 in Step 5
print("repeat check:", check_handoff(A_ATTEMPTS, "order", "refund", {"max_handoffs": 9, "repeat_check": True}))

None
2
run A route: 2
repeat check: the same handoff again


## Recap
- A loop is a handoff cycle with no new facts, usually from a missing fact that each agent asks the other for; lost
  context is a note that leaves out what the next agent needs.
- Write every note so it stands on its own.
- Three guards stop a run: a call budget, a handoff limit and a repeat check on the route. Each stop returns a
  reason and hands the case to a person.
- Count the calls with a log, and test the guards on a case that loops on purpose.

**Interview:** "How do you stop two agents from passing a case back and forth?" Strong answer: "I put the stop in code,
not in the prompt: a call budget checked before each call, a handoff limit, and a repeat check on the route. Each stop
escalates to a person with the route and the reason. I test it with a case that loops on purpose."